# BERT QA - Version 1 (SQuAD v2)

Objectif: fine-tuning de **bert-base-uncased** pour améliorer fortement EM/F1.


## 1) Installation des dépendances


In [50]:
# Décommente si besoin:
%pip install -q transformers datasets evaluate accelerate sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [51]:
import datasets, evaluate, transformers

print(datasets.__version__, evaluate.__version__, transformers.__version__)

import sys, torch

print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("MPS:", torch.backends.mps.is_available())

from datasets import load_dataset

print("datasets import OK")
ds = load_dataset("squad_v2", split="validation[:50]")
print("dataset OK", len(ds))

from transformers import AutoTokenizer, AutoModelForQuestionAnswering

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
mdl = AutoModelForQuestionAnswering.from_pretrained("bert-base-uncased")
print("BERT load OK")


4.6.1 0.4.6 5.2.0
Python: /Users/albanecoiffe/Downloads/NLP/.venv/bin/python
Torch: 2.10.0
MPS: True
datasets import OK
dataset OK 50


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2411.88it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]              
BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can 

BERT load OK


## 2) Imports et device (Apple MPS)


In [52]:
import os
import json
import numpy as np
import torch

from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)


Device: mps


## 3) Configuration


In [53]:
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 384
DOC_STRIDE = 128

TRAIN_SAMPLES = 20000   # None pour full train
DEV_SAMPLES = 3000      # None pour full validation

BATCH_SIZE = 8
EPOCHS = 2
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
OUTPUT_DIR = "bert_qa_outputs"

SEED = 42


## 4) Chargement dataset SQuAD v2


In [54]:
raw_datasets = load_dataset("squad_v2")

if TRAIN_SAMPLES is not None:
    raw_datasets["train"] = raw_datasets["train"].select(range(min(TRAIN_SAMPLES, len(raw_datasets["train"]))))
if DEV_SAMPLES is not None:
    raw_datasets["validation"] = raw_datasets["validation"].select(range(min(DEV_SAMPLES, len(raw_datasets["validation"]))))

print(raw_datasets)


DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 3000
    })
})


## 5) Tokenizer et préprocessing train


In [55]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def preprocess_train(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    offset_mapping = inputs.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = inputs["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = inputs.sequence_ids(i)
        sample_idx = sample_map[i]
        answers = examples["answers"][sample_idx]

        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                token_start_index += 1
            start_positions.append(token_start_index - 1)

            while offsets[token_end_index][1] >= end_char:
                token_end_index -= 1
            end_positions.append(token_end_index + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs


## 6) Préprocessing validation (avec offsets pour post-process)


In [56]:
def preprocess_validation(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)]

    inputs["example_id"] = example_ids
    return inputs


## 7) Tokenization datasets


In [57]:
tokenized_train = raw_datasets["train"].map(
    preprocess_train,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

tokenized_val = raw_datasets["validation"].map(
    preprocess_validation,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)

print(tokenized_train)
print(tokenized_val)


Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 20158
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 3007
})


## 8) Modèle BERT QA


In [58]:
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
model.to(device)
print(model.__class__.__name__)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2848.47it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]              
BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can 

BertForQuestionAnswering


## 9) Post-processing des prédictions + métriques SQuAD v2


In [ ]:
metric = evaluate.load("squad_v2")


def postprocess_qa_predictions(examples, features, raw_predictions, n_best_size=20, max_answer_length=30):
    all_start_logits, all_end_logits = raw_predictions

    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = {}
    for i, f in enumerate(features):
        ex_id = f["example_id"]
        features_per_example.setdefault(example_id_to_index[ex_id], []).append(i)

    predictions = []

    for example_index, example in enumerate(examples):
        feature_indices = features_per_example.get(example_index, [])
        context = example["context"]

        min_null_score = None
        valid_answers = []

        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]

            cls_score = start_logits[0] + end_logits[0]
            if min_null_score is None or min_null_score < cls_score:
                min_null_score = cls_score

            start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
            end_indexes = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()

            for start_index in start_indexes:
                for end_index in end_indexes:
                    if start_index >= len(offset_mapping) or end_index >= len(offset_mapping):
                        continue
                    if offset_mapping[start_index] is None or offset_mapping[end_index] is None:
                        continue
                    if end_index < start_index or end_index - start_index + 1 > max_answer_length:
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    valid_answers.append({
                        "score": start_logits[start_index] + end_logits[end_index],
                        "text": context[start_char:end_char],
                    })

        if len(valid_answers) > 0:
            best_answer = sorted(valid_answers, key=lambda x: x["score"], reverse=True)[0]
            if min_null_score is not None and min_null_score > best_answer["score"]:
                predictions.append({"id": example["id"], "prediction_text": "", "no_answer_probability": 1.0})
            else:
                predictions.append({"id": example["id"], "prediction_text": best_answer["text"], "no_answer_probability": 0.0})
        else:
            predictions.append({"id": example["id"], "prediction_text": "", "no_answer_probability": 1.0})

    return predictions


def compute_metrics(eval_preds):
    predictions = postprocess_qa_predictions(
        raw_datasets["validation"],
        tokenized_val,
        eval_preds,
    )
    references = [{"id": ex["id"], "answers": ex["answers"]} for ex in raw_datasets["validation"]]
    return metric.compute(predictions=predictions, references=references)


## 10) TrainingArguments + Trainer


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=False,
    push_to_hub=False,
    logging_steps=100,
    report_to=[],
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)



## 11) Entraînement


In [ ]:
train_result = trainer.train()
print(train_result)


## 12) Évaluation finale


In [ ]:
metrics = trainer.evaluate()
print("Final metrics:")
for k, v in metrics.items():
    print(f"{k}: {v}")


## 13) Sauvegarde modèle et tokenizer


In [ ]:
save_dir = "bert_qa_best"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved to:", save_dir)
